# メタ評価記事の動作検証ノートブック

LLMジャッジの信頼性を、人間ラベルとの一致率 (Cohen's kappa) で測るメタ評価の動作確認ノートブックです。

## このノートブックで確認すること
1. パッケージインストールと基本セットアップ
2. 前回記事のLLMジャッジ3つを再定義
3. 人間ラベル付き評価データセット (20件) の準備
4. ジャッジ実行と判定値の収集
5. 単純一致率と Cohen's kappa の計算
6. 結果の解釈

## 想定環境
- Databricks ワークスペース (有償版)
- アプリLLM: `databricks-meta-llama-3-3-70b-instruct`
- ジャッジLLM: `databricks-claude-sonnet-4-6` (アプリとは別系統、Self-Enhancement Bias対策)
- 実行時間: 5〜10分

## Step 0: パッケージインストール

In [0]:
%pip install --upgrade databricks-agents mlflow scikit-learn
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 110.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.4/611.4 kB 23.0 MB/s eta 0:00:00
  Attempting uninstall: blinker
    Found existing installation: blinker 1.7.0
    Not uninstalling blinker at /usr/lib/python3/dist-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-d0754881-efd4-4176-b902-0805f3e024f7
    Can't uninstall 'blinker'. No files were found to uninstall.
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Not uninstalling scikit-learn at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs

## Step 1: 基本セットアップ

In [0]:
import json
import os
import mlflow
import pandas as pd
from databricks.sdk import WorkspaceClient

# rate limit対策 (有償版なのでやや緩めに)
os.environ["MLFLOW_GENAI_EVAL_MAX_WORKERS"] = "4"
os.environ["MLFLOW_GENAI_EVAL_MAX_SCORER_WORKERS"] = "4"

CATALOG = "takaakiyayoi_catalog"
SCHEMA = "llmops_meta_eval"
PROMPT_NAME = f"{CATALOG}.{SCHEMA}.triage_assistant"

w = WorkspaceClient()
client = w.serving_endpoints.get_open_ai_client()
# アプリ側のLLM (トリアージ応答を生成)
APP_MODEL_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
# ジャッジ側のLLM (アプリ応答を評価)。アプリLLMと別系統を選ぶことで
# Self-Enhancement Bias (自己選好バイアス) の影響を下げる
JUDGE_MODEL_URI = "databricks:/databricks-claude-sonnet-4-6"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

print(f"PROMPT_NAME: {PROMPT_NAME}")
print(f"APP_MODEL_ENDPOINT: {APP_MODEL_ENDPOINT}")
print(f"JUDGE_MODEL_URI: {JUDGE_MODEL_URI}")
print(f"✅ Step 1 完了")

PROMPT_NAME: takaakiyayoi_catalog.llmops_meta_eval.triage_assistant
APP_MODEL_ENDPOINT: databricks-meta-llama-3-3-70b-instruct
JUDGE_MODEL_URI: databricks:/databricks-claude-sonnet-4-6
✅ Step 1 完了


## Step 2: ジャッジを再定義

前回のジャッジ設計記事で作った3つのLLMジャッジを再定義します。Python scorer (schema_compliance) は決定論的なので、メタ評価の対象外とします。

In [0]:
from mlflow.genai.judges import make_judge

hallucination_judge = make_judge(
    name="no_hallucination",
    instructions=(
        "応答 ({{ outputs }}) が、問い合わせ ({{ inputs }}) に対して存在しない機能・"
        "サービス・手続きを断定的に案内していないか評価する。"
        "存在しない機能を肯定的に案内している場合は 'fail'、"
        "存在を確認できない要素を慎重に扱っているか正規の手続きを案内している場合は 'pass' とする。"
        "判定根拠を1-2文で述べる。"
    ),
    model=JUDGE_MODEL_URI,
)

intent_judge = make_judge(
    name="intent_match",
    instructions=(
        "応答 ({{ outputs }}) が、問い合わせ ({{ inputs }}) の本筋に答えているか評価する。"
        "expectations.must_address_actual_request に書かれた本筋に応答が言及していれば 'pass'、"
        "別のトピックにすり替わっていれば 'fail' とする。"
        "判定根拠を1-2文で述べる。"
    ),
    model=JUDGE_MODEL_URI,
)

politeness_judge = make_judge(
    name="politeness",
    instructions=(
        "応答 ({{ outputs }}) の draft_reply が、相手に寄り添う表現を含むか評価する。"
        "事務的な情報伝達のみで「お手数ですが」「ご不便をおかけし」等の配慮表現が一切ない場合は 'fail'、"
        "配慮や共感を示す表現が含まれていれば 'pass' とする。"
        "判定根拠を1-2文で述べる。"
    ),
    model=JUDGE_MODEL_URI,
)

print(f"✅ 3つのLLMジャッジ定義完了 (judge model: {JUDGE_MODEL_URI})")

✅ 3つのLLMジャッジ定義完了 (judge model: databricks:/databricks-claude-sonnet-4-6)


## Step 3: triage プロンプトの登録と関数の準備

新規スキーマ前提なので、最初にv1プロンプトを登録します。既に存在する場合は新バージョンが追加されます。

In [0]:
PROMPT_V1 = """あなたは顧客サポートのトリアージ担当です。
以下の問い合わせを分類し、JSON形式で出力してください。

出力スキーマ:
{
  "category": "billing | technical | account | other",
  "priority": "高 | 中 | 低",
  "draft_reply": "回答案 (200文字以内)"
}

問い合わせ:
{{inquiry}}
"""

# productionエイリアスが既に存在するか確認
try:
    existing = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@production")
    print(f"✅ 既存のproductionエイリアスあり: version={existing.version}")
except Exception:
    print("productionエイリアスが見つかりません。v1を新規登録します。")
    prompt_v1 = mlflow.genai.register_prompt(
        name=PROMPT_NAME, template=PROMPT_V1, commit_message="v1: トリアージプロンプト"
    )
    mlflow.genai.set_prompt_alias(
        name=PROMPT_NAME, alias="production", version=prompt_v1.version
    )
    print(f"✅ プロンプト登録成功: version={prompt_v1.version}")

productionエイリアスが見つかりません。v1を新規登録します。
✅ プロンプト登録成功: version=1


In [0]:
@mlflow.trace
def triage(inquiry: str) -> dict:
    prompt_obj = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@production")
    prompt = prompt_obj.format(inquiry=inquiry)
    response = client.chat.completions.create(
        model=APP_MODEL_ENDPOINT,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)


# 1件流して動作確認
sample = triage("ログインできません。")
print(json.dumps(sample, ensure_ascii=False, indent=2))

{
  "category": "account",
  "priority": "高",
  "draft_reply": "ログインに問題が発生している場合は、パスワードのリセットやアカウントの確認を試してください。詳細はお聞きします。"
}


Trace(trace_id=tr-0f5fe42b8af52a5366fa40d9b5b3bc52)

## Step 4: 人間ラベル付きデータセットの準備

20件の問い合わせに対し、各ジャッジが本来 pass/fail を返すべきかを「ドメイン専門家のレビュー結果」として直接コードで定義します。実運用では Databricks の Labeling Sessions を使うのが推奨です。

各レコードには、3つのジャッジそれぞれに対する人間判定を `pass`/`fail`/`None` (該当しない場合) で記載します。

In [0]:
# 20件の人間ラベル付き評価データ
# 設計方針: 各観点 (no_hallucination / intent_match / politeness) に対し、
# 全件で人間判定を pass/fail で付与する。None は使わない。
# これにより各観点ごとに同じサンプル数 (n=20) で kappa が計算できる。
human_labeled = [
    {
        "inquiry": "ログインできない状況が続いています。ご対応をお願いします。",
        "human_no_hallucination": "pass",
        "human_intent_match": "pass",
        "human_politeness": "pass",
        "intent_target": "ログイン問題への対応",
    },
    {
        "inquiry": "アカウント情報を変更したいのですが、手順を教えていただけますでしょうか。",
        "human_no_hallucination": "pass",
        "human_intent_match": "pass",
        "human_politeness": "pass",
        "intent_target": "アカウント情報変更の手順",
    },
    {
        "inquiry": "料金プランの変更について教えてください。",
        "human_no_hallucination": "pass",
        "human_intent_match": "pass",
        "human_politeness": "pass",
        "intent_target": "料金プラン変更の案内",
    },
    {
        "inquiry": "パスワードリセットのやり方は。",
        "human_no_hallucination": "pass",
        "human_intent_match": "pass",
        "human_politeness": "fail",
        "intent_target": "パスワードリセットの方法",
    },
    {
        "inquiry": "請求書ほしい。",
        "human_no_hallucination": "pass",
        "human_intent_match": "pass",
        "human_politeness": "fail",
        "intent_target": "請求書の提供",
    },
    {
        "inquiry": "やり方教えて。",
        "human_no_hallucination": "pass",
        "human_intent_match": "pass",
        "human_politeness": "fail",
        "intent_target": "手順の案内",
    },
    {
        "inquiry": "すぐ送って。",
        "human_no_hallucination": "pass",
        "human_intent_match": "pass",
        "human_politeness": "fail",
        "intent_target": "資料の送付",
    },
    {
        "inquiry": "プレミアム会員専用のシークレットモードの解除方法を教えてください。",
        "human_no_hallucination": "pass",
        "human_intent_match": "pass",
        "human_politeness": "pass",
        "intent_target": "機能の解除方法",
    },
    {
        "inquiry": "カスタマーポータルのワンクリック返金ボタンが見つかりません。",
        "human_no_hallucination": "pass",
        "human_intent_match": "pass",
        "human_politeness": "pass",
        "intent_target": "返金手続きの案内",
    },
    {
        "inquiry": "ターボブースト機能の有効化方法を教えてください。",
        "human_no_hallucination": "pass",
        "human_intent_match": "pass",
        "human_politeness": "pass",
        "intent_target": "機能の有効化方法",
    },
    {
        "inquiry": "メガセーブモードはどこから設定しますか?",
        "human_no_hallucination": "pass",
        "human_intent_match": "pass",
        "human_politeness": "pass",
        "intent_target": "機能の設定方法",
    },
    {
        "inquiry": "登録メールアドレスの変更方法を教えてください。",
        "human_no_hallucination": "pass",
        "human_intent_match": "pass",
        "human_politeness": "pass",
        "intent_target": "メールアドレス変更の方法",
    },
    {
        "inquiry": "請求書のダウンロード方法を教えてください。",
        "human_no_hallucination": "pass",
        "human_intent_match": "pass",
        "human_politeness": "pass",
        "intent_target": "請求書のダウンロード方法",
    },
    {
        "inquiry": "先日キャンセルした注文の請求書だけ再発行してほしいです。",
        "human_no_hallucination": "pass",
        "human_intent_match": "pass",
        "human_politeness": "pass",
        "intent_target": "請求書の再発行",
    },
    {
        "inquiry": "解約はしたくないのですが料金プランだけ見直したいです。",
        "human_no_hallucination": "pass",
        "human_intent_match": "pass",
        "human_politeness": "pass",
        "intent_target": "料金プランの見直し",
    },
    {
        "inquiry": "ログインができず困っているので、メールアドレスを変更したいです。",
        "human_no_hallucination": "pass",
        "human_intent_match": "pass",
        "human_politeness": "pass",
        "intent_target": "ログイン問題の解決",
    },
    {
        "inquiry": "請求書が必要ですが、支払い方法の確認もしたいです。",
        "human_no_hallucination": "pass",
        "human_intent_match": "pass",
        "human_politeness": "pass",
        "intent_target": "請求書の提供と支払い方法の案内",
    },
    {
        "inquiry": "キャンセル方法と、その後の返金について教えてください。",
        "human_no_hallucination": "pass",
        "human_intent_match": "pass",
        "human_politeness": "pass",
        "intent_target": "キャンセル方法と返金プロセス",
    },
    {
        "inquiry": "解約したい。返金は?",
        "human_no_hallucination": "pass",
        "human_intent_match": "pass",
        "human_politeness": "fail",
        "intent_target": "解約手続きと返金案内",
    },
    {
        "inquiry": "至急対応してください、本当に困っています。",
        "human_no_hallucination": "pass",
        "human_intent_match": "pass",
        "human_politeness": "pass",
        "intent_target": "至急の対応",
    },
]

print(f"✅ 人間ラベル付きデータ: {len(human_labeled)} 件")

# ラベルの分布を確認
for col in ["human_no_hallucination", "human_intent_match", "human_politeness"]:
    counts = pd.Series([r[col] for r in human_labeled]).value_counts().to_dict()
    print(f"  {col}: {counts}")

✅ 人間ラベル付きデータ: 20 件
  human_no_hallucination: {'pass': 20}
  human_intent_match: {'pass': 20}
  human_politeness: {'pass': 15, 'fail': 5}


## Step 5: トリアージ実行とジャッジ実行

各レコードに対して triage を実行し、得られた出力に対してジャッジを実行します。`mlflow.genai.evaluate` を使って、ジャッジ判定をまとめて取得します。

In [0]:
# evaluate に渡せる形式に変換
eval_data = []
for rec in human_labeled:
    eval_data.append(
        {
            "inputs": {"inquiry": rec["inquiry"]},
            "expectations": {"must_address_actual_request": rec["intent_target"]},
        }
    )


@mlflow.trace
def triage_for_eval(inquiry: str) -> dict:
    prompt_obj = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@production")
    prompt = prompt_obj.format(inquiry=inquiry)
    response = client.chat.completions.create(
        model=APP_MODEL_ENDPOINT,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)


results = mlflow.genai.evaluate(
    data=eval_data,
    predict_fn=triage_for_eval,
    scorers=[hallucination_judge, intent_judge, politeness_judge],
)
print(f"✅ evaluate 完了: Run ID = {results.run_id}")

2026/05/18 05:10:14 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/05/18 05:10:14 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/20 [Elapsed: 00:00, Remaining: ?]

✅ evaluate 完了: Run ID = 3daf2c35234e44d0822b6d50e760c3a2


## Step 6: ジャッジ判定の取得

評価ランからジャッジ判定を取り出し、人間ラベルと並べた DataFrame を作ります。

In [0]:
eval_traces = mlflow.search_traces(run_id=results.run_id)
print(f"トレース取得: {len(eval_traces)} 件")
print(f"列名: {list(eval_traces.columns)}")

トレース取得: 20 件
列名: ['trace_id', 'trace', 'client_request_id', 'state', 'request_time', 'execution_duration', 'request', 'response', 'trace_metadata', 'tags', 'spans', 'assessments']


In [0]:
JUDGE_NAMES = {"no_hallucination", "intent_match", "politeness"}


def extract_judge_scores(row):
    """assessments から各ジャッジの判定値を取り出す"""
    scores = {name: None for name in JUDGE_NAMES}
    assessments = row.get("assessments")
    if not isinstance(assessments, list):
        return scores
    for a in assessments:
        if not isinstance(a, dict):
            continue
        name = a.get("assessment_name")
        if name not in JUDGE_NAMES:
            continue
        feedback = a.get("feedback") or {}
        value = feedback.get("value")
        error = feedback.get("error")
        if error is not None:
            scores[name] = "ERROR"
        else:
            scores[name] = value
    return scores


def extract_inquiry(row):
    req = row.get("request") or {}
    if isinstance(req, str):
        try:
            req = json.loads(req)
        except Exception:
            req = {}
    return req.get("inquiry", "") if isinstance(req, dict) else ""


# ジャッジ判定を抽出
judge_rows = []
for _, row in eval_traces.iterrows():
    inquiry = extract_inquiry(row)
    scores = extract_judge_scores(row)
    judge_rows.append({"inquiry": inquiry, **scores})

judge_df = pd.DataFrame(judge_rows)

# 人間ラベルとマージ
human_df = pd.DataFrame(
    [
        {
            "inquiry": rec["inquiry"],
            "human_no_hallucination": rec["human_no_hallucination"],
            "human_intent_match": rec["human_intent_match"],
            "human_politeness": rec["human_politeness"],
        }
        for rec in human_labeled
    ]
)

merged = human_df.merge(judge_df, on="inquiry", how="left")
print(f"マージ後の件数: {len(merged)}")
print(f"列: {list(merged.columns)}")
display(merged)

マージ後の件数: 20
列: ['inquiry', 'human_no_hallucination', 'human_intent_match', 'human_politeness', 'no_hallucination', 'intent_match', 'politeness']


inquiry,human_no_hallucination,human_intent_match,human_politeness,no_hallucination,intent_match,politeness
ログインできない状況が続いています。ご対応をお願いします。,pass,pass,pass,pass,pass,pass
アカウント情報を変更したいのですが、手順を教えていただけますでしょうか。,pass,pass,pass,pass,pass,fail
料金プランの変更について教えてください。,pass,pass,pass,pass,pass,fail
パスワードリセットのやり方は。,pass,pass,fail,pass,pass,fail
請求書ほしい。,pass,pass,fail,pass,pass,fail
やり方教えて。,pass,pass,fail,pass,pass,fail
すぐ送って。,pass,pass,fail,pass,fail,pass
プレミアム会員専用のシークレットモードの解除方法を教えてください。,pass,pass,pass,fail,pass,fail
カスタマーポータルのワンクリック返金ボタンが見つかりません。,pass,pass,pass,fail,fail,pass
ターボブースト機能の有効化方法を教えてください。,pass,pass,pass,fail,pass,fail


## Step 7: Cohen's kappa の計算

ジャッジごとに単純一致率と Cohen's kappa を計算します。人間ラベルが None のレコードは該当ジャッジの集計から除外します。

In [0]:
from sklearn.metrics import cohen_kappa_score


def compute_agreement(df, human_col, judge_col):
    """人間ラベルとジャッジ判定の一致率と Cohen's kappa を計算"""
    # None と ERROR を除外
    valid = df[df[human_col].notna() & df[judge_col].notna() & (df[judge_col] != "ERROR")].copy()
    n = len(valid)
    if n < 2:
        return {"n": n, "simple_agreement": None, "cohens_kappa": None}

    # pass/fail を 1/0 にエンコード
    valid["human_enc"] = (valid[human_col] == "pass").astype(int)
    valid["judge_enc"] = (valid[judge_col] == "pass").astype(int)

    simple = (valid["human_enc"] == valid["judge_enc"]).mean()
    try:
        kappa = cohen_kappa_score(valid["human_enc"], valid["judge_enc"])
    except Exception as e:
        kappa = None
        print(f"  kappa 計算エラー: {e}")

    # 内訳も計算
    confusion = pd.crosstab(valid["human_enc"], valid["judge_enc"], rownames=["human"], colnames=["judge"])

    return {
        "n": n,
        "simple_agreement": round(simple, 3),
        "cohens_kappa": round(kappa, 3) if kappa is not None else None,
        "confusion": confusion,
    }


judges_to_check = [
    ("no_hallucination", "human_no_hallucination"),
    ("intent_match", "human_intent_match"),
    ("politeness", "human_politeness"),
]

summary_rows = []
for judge_name, human_col in judges_to_check:
    result = compute_agreement(merged, human_col, judge_name)
    summary_rows.append({
        "judge": judge_name,
        "n": result["n"],
        "simple_agreement": result["simple_agreement"],
        "cohens_kappa": result["cohens_kappa"],
    })
    print(f"\n=== {judge_name} ===")
    print(f"  n = {result['n']}")
    print(f"  単純一致率 = {result['simple_agreement']}")
    print(f"  Cohen's kappa = {result['cohens_kappa']}")
    if "confusion" in result and result["confusion"] is not None:
        print(f"  混同行列:")
        print(result["confusion"])

summary_df = pd.DataFrame(summary_rows)
print("\n=== サマリ ===")
display(summary_df)


=== no_hallucination ===
  n = 20
  単純一致率 = 0.75
  Cohen's kappa = 0.0
  混同行列:
judge  0   1
human       
1      5  15

=== intent_match ===
  n = 20
  単純一致率 = 0.9
  Cohen's kappa = 0.0
  混同行列:
judge  0   1
human       
1      2  18

=== politeness ===
  n = 20
  単純一致率 = 0.45
  Cohen's kappa = 0.083
  混同行列:
judge   0  1
human       
0       4  1
1      10  5

=== サマリ ===


judge,n,simple_agreement,cohens_kappa
no_hallucination,20,0.75,0.0
intent_match,20,0.9,0.0
politeness,20,0.45,0.083


## Step 8: 結果の解釈ガイド

Cohen's kappa の解釈は Landis & Koch (1977) の目安が広く使われています。

| kappa 値 | 解釈 |
| --- | --- |
| 0.81 - 1.00 | ほぼ完全な一致 |
| 0.61 - 0.80 | 実質的な一致 |
| 0.41 - 0.60 | 中程度の一致 |
| 0.21 - 0.40 | 弱い一致 |
| 0.00 - 0.20 | ほんのわずかな一致 |
| 0.00 未満 | 偶然以下 (実質一致なし) |

本番運用のジャッジとして使うなら、Cohen's kappa が 0.6 以上であることが目安になります。それ未満なら、instructions の見直しや評価データセットの精査が必要です。

## 検証完了チェックリスト

- [ ] Step 5 の evaluate が成功 (scorer invocation failed が 0/20 か少数)
- [ ] Step 6 でマージされた DataFrame に各ジャッジの判定値が入っている (null がない)
- [ ] Step 7 で 3つのジャッジそれぞれの kappa 値が計算できた

### 想定される結果の解釈

人間ラベルとジャッジ判定の組み合わせは観点ごとに次のようになると予想されます。

- **no_hallucination**: 人間ラベルもジャッジも全件 pass の可能性が高く、単一クラスのため kappa が `nan` または 0 になる可能性。「メタ評価には判定の分散が必要」という学びになる
- **intent_match**: 同様に人間ラベルが全件 pass で、ジャッジは「すぐ送って」など一部 fail の可能性。人間側が単一クラスのため kappa は 0 付近になる可能性
- **politeness**: 配慮表現の有無で人間ラベルもジャッジも分散があり、kappa が綺麗に計算できる。kappa の値次第で「ジャッジが厳しすぎる」「人間とジャッジの基準がずれている」などの議論ができる

記事ではこの結果を踏まえ「メタ評価が機能する観点と機能しない観点がある」というメッセージで構成します。